# Step 4: Provider-Level Feature Engineering

Goal: aggregate the cleaned claim-level table up to **one row per provider**, engineer features that capture billing behavior, claim complexity, and beneficiary profile, then formally test which features are significant and check for multicollinearity before locking in a feature set for modeling.

Two aggregation-design decisions matter throughout this notebook:
1. **Claim-level features** (volume, amounts, code complexity) are aggregated directly from the claim-level table.
2. **Beneficiary demographic features** (age, chronic conditions) are aggregated from a table deduplicated to one row per (Provider, BeneID) first — otherwise a beneficiary with many visits gets their demographics counted multiple times, silently biasing the provider's profile toward whichever beneficiary visits most.

No feature uses `PotentialFraud` as an input anywhere — only as the final label, to avoid leaking target information into the feature set.

In [1]:
import sys
sys.path.append('..')

from src.data_loader import build_claim_level_dataset
from src.cleaning import clean_claim_level_dataset
from src.features import build_provider_features
from src.feature_selection import evaluate_feature_significance, flag_weak_features, compute_vif, iterative_vif_selection

df = clean_claim_level_dataset(build_claim_level_dataset())
provider_features = build_provider_features(df)
provider_features.shape

(5410, 20)

**What this cell does:** runs the full feature engineering pipeline from `src/features.py`, combining six feature blocks (claim volume, claim amounts, diagnosis/procedure complexity, physician involvement, beneficiary demographics, claim/stay duration) into one provider-level table.

**Expected output:** `(5410, 19)` — one row per provider, `Provider` + `PotentialFraud` + 17 engineered features.

In [2]:
provider_features.isna().sum()[provider_features.isna().sum() > 0]

Series([], dtype: int64)

**What this checks:** confirms no unexpected missing values slipped through the aggregation. If anything shows up here, it likely means a provider had zero claims of a certain type (e.g. `AvgLengthOfStayDays` is already handled inside `build_duration_features()` for providers with no inpatient claims — filled with 0, not left as NaN).

In [3]:
provider_features.to_csv('../data/processed/provider_features.csv', index=False)

**What this does:** saves the full provider-level feature table so Step 5 (train/test split) can load it directly. Not committed to git — regenerate locally by re-running this notebook.

## Feature Significance

Before finalizing the feature set, check which features actually separate Fraud from Non-Fraud providers — both statistically (p-value) and practically (effect size). At n=5,410, p-values alone can make almost any feature look "significant," so effect size (Cohen's d) is what actually distinguishes a strong feature from noise.

In [4]:
feature_cols = [c for c in provider_features.columns if c not in ('Provider', 'PotentialFraud')]

sig_df = evaluate_feature_significance(provider_features, feature_cols)
sig_df

,feature,mean_fraud,mean_non_fraud,t_statistic,p_value,cohens_d
0,MaxClaimAmt,43447.332016,9874.863377,29.143041,4.949565e-113,1.605940
1,AvgLengthOfStayDays,5.316086,1.868263,28.314255,8.079945e-117,1.205726
2,StdClaimAmt,6376.770794,2016.201421,21.338225,1.786102e-74,1.110047
3,PropInpatient,0.329196,0.125518,13.792328,1.108756e-37,0.679530
4,NumClaims,420.545455,70.435359,10.879004,6.352459e-25,0.674430
5,NumUniqueOperatingPhysicians,19.286561,5.494902,11.086022,9.296198e-26,0.670120
6,AvgNumDiagnosisCodes,4.428544,3.301988,13.452697,3.062471e-36,0.645406
7,AvgClaimAmt,3842.795104,1523.780804,13.160883,7.245999e-35,0.644057
8,AvgNumProcedureCodes,0.252301,0.093123,13.094118,1.315985e-34,0.633396
9,NumUniqueAttendingPhysicians,41.114625,13.682912,10.586436,7.479027e-24,0.625269


**How to read this:** sorted by `|Cohen's d|`, largest effect first. Features near the top (likely `NumClaims`, `MedianClaimAmt`/`AvgClaimAmt`, `PropInpatient`) confirm what the EDA boxplots in Step 3 already suggested — fraudulent providers bill more, more often, and lean more inpatient. Features near the bottom (small `|Cohen's d|`, high p-value) are candidates for the `flag_weak_features()` check below.

In [5]:
flagged = flag_weak_features(sig_df)
flagged[flagged['is_weak']][['feature', 'p_value', 'cohens_d']]

,feature,p_value,cohens_d
14,AvgNumPhysiciansInvolved,0.000002,0.156819
15,PropDeceasedBeneficiaries,0.074594,0.053463
16,DeductibleToReimbursementRatio,0.174900,-0.027401
17,AvgAgeOfBeneficiaries,0.925385,0.002888


**Important caveat:** a feature flagged "weak" here isn't automatically useless — this test only detects **linear, mean-difference** signal. A feature could still matter to a tree-based model (Random Forest, XGBoost) through nonlinear splits or interactions a t-test can't see. Treat this as a first-pass filter to discuss and justify, not an automatic drop list — for the project report, being able to say "we tested X and found it statistically weak on its own" is itself a useful, defensible finding.

## Multicollinearity Check (Variance Inflation Factor)

VIF matters mainly for linear models like logistic regression, which assume independent predictors — highly correlated features destabilize their coefficients. Tree-based models are largely unaffected by multicollinearity, so this check specifically informs which feature set to use with which model.

In [6]:
vif_df = compute_vif(provider_features, feature_cols)
vif_df

,feature,VIF
0,AvgClaimAmt,35.704085
1,MedianClaimAmt,19.005337
2,PropInpatient,11.625865
3,NumUniqueOperatingPhysicians,10.133402
4,NumUniqueAttendingPhysicians,9.081358
5,StdClaimAmt,8.681072
6,AvgNumDiagnosisCodes,6.849888
7,AvgNumProcedureCodes,5.309228
8,MaxClaimAmt,4.623807
9,AvgLengthOfStayDays,2.263817


**What this shows:** the remaining VIF values (after earlier fixes removed exact arithmetic identities like `TotalClaimAmt = NumClaims × AvgClaimAmt`) reflect *genuine* domain correlation, not construction errors — e.g. `AvgClaimAmt`, `MedianClaimAmt`, `StdClaimAmt`, and `MaxClaimAmt` are four different summary statistics of the same claim-amount distribution, so they naturally move together. `PropInpatient` correlates with claim amount because inpatient claims cost more (the same 0.84 correlation found in Step 3's EDA). None of this is wrong — it just means logistic regression needs a decorrelated subset.

In [7]:
logistic_features, elimination_log = iterative_vif_selection(provider_features, feature_cols, threshold=10)

print("Final features for logistic regression:")
print(logistic_features)
print("\nElimination log:")
elimination_log

Final features for logistic regression:
['NumClaims', 'MedianClaimAmt', 'StdClaimAmt', 'MaxClaimAmt', 'DeductibleToReimbursementRatio', 'AvgNumDiagnosisCodes', 'AvgNumProcedureCodes', 'AvgNumPhysiciansInvolved', 'NumUniqueAttendingPhysicians', 'AvgAgeOfBeneficiaries', 'PropDeceasedBeneficiaries', 'AvgChronicConditions', 'AvgClaimsPerBeneficiary', 'AvgClaimDurationDays', 'AvgLengthOfStayDays']

Elimination log:


,dropped_feature,vif_at_drop
0,AvgClaimAmt,35.704085
1,PropInpatient,11.548555
2,NumUniqueOperatingPhysicians,10.132010


**What this does:** greedily drops the single highest-VIF feature, recomputes, and repeats until every remaining feature is below the threshold (10). The `elimination_log` records what was dropped and why — useful evidence for the report that feature selection was deliberate, not arbitrary.

**Result:** `AvgClaimAmt` dropped first (highest VIF, but its information isn't lost — `MedianClaimAmt`/`StdClaimAmt`/`MaxClaimAmt` remain), then `PropInpatient` (the known 0.84 correlation with claim amount), then `NumUniqueOperatingPhysicians` (redundant with `NumUniqueAttendingPhysicians`).

## Two Feature Sets Going Forward

This is a deliberate modeling choice, not a compromise — different algorithms have different assumptions about their inputs:

| Model type | Feature set | Why |
|---|---|---|
| Logistic Regression | `logistic_features` (15 features, VIF < 10) | Assumes independent predictors; multicollinearity destabilizes coefficients |
| Random Forest / Boosting | `feature_cols` (full 17 features) | Not sensitive to multicollinearity; dropping informative features unnecessarily could lose signal |

Both feature sets are saved below for use in Step 5 onward.

In [8]:
import json

feature_sets = {
    'full_features': feature_cols,
    'logistic_features': logistic_features,
}

with open('../data/processed/feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=2)

feature_sets

{'full_features': ['NumClaims',
  'PropInpatient',
  'AvgClaimAmt',
  'MedianClaimAmt',
  'StdClaimAmt',
  'MaxClaimAmt',
  'DeductibleToReimbursementRatio',
  'AvgNumDiagnosisCodes',
  'AvgNumProcedureCodes',
  'AvgNumPhysiciansInvolved',
  'NumUniqueAttendingPhysicians',
  'NumUniqueOperatingPhysicians',
  'AvgAgeOfBeneficiaries',
  'PropDeceasedBeneficiaries',
  'AvgChronicConditions',
  'AvgClaimsPerBeneficiary',
  'AvgClaimDurationDays',
  'AvgLengthOfStayDays'],
 'logistic_features': ['NumClaims',
  'MedianClaimAmt',
  'StdClaimAmt',
  'MaxClaimAmt',
  'DeductibleToReimbursementRatio',
  'AvgNumDiagnosisCodes',
  'AvgNumProcedureCodes',
  'AvgNumPhysiciansInvolved',
  'NumUniqueAttendingPhysicians',
  'AvgAgeOfBeneficiaries',
  'PropDeceasedBeneficiaries',
  'AvgChronicConditions',
  'AvgClaimsPerBeneficiary',
  'AvgClaimDurationDays',
  'AvgLengthOfStayDays']}

**What this does:** saves both feature lists to `data/processed/feature_sets.json`, so Step 5 (train/test split) and Step 6 (modeling) can load the appropriate list by name instead of hardcoding column names in multiple places.

## Summary of Step 4

- Built a **provider-level feature table**: 5,410 rows × 17 engineered features across 6 categories (claim volume, claim amounts, diagnosis/procedure complexity, physician involvement, beneficiary demographics, claim/stay duration)
- **No target leakage:** no feature uses `PotentialFraud` as an input
- **Significance testing:** ranked features by effect size (Cohen's d), not p-value alone, since p-values are near-automatically significant at n=5,410
- **Multicollinearity resolved in two passes:**
  1. Removed exact arithmetic identities (e.g. `TotalClaimAmt = NumClaims × AvgClaimAmt`, `NumInpatientClaims`/`NumOutpatientClaims` vs `NumClaims`/`PropInpatient`) directly in `features.py`, so they can't resurface on re-runs
  2. Used iterative VIF elimination for genuine (non-identity) correlations, producing a 15-feature decorrelated set for logistic regression
- **Two feature sets saved** (`feature_sets.json`): full set for tree-based/boosting models, VIF-clean subset for logistic regression
- **Strongest features so far** (highest effect size): claim volume (`NumClaims`) and claim amount statistics — consistent with Step 3's EDA findings